In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr=1,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.08171672374010086
epoch:  1, loss: 0.08092395961284637
epoch:  2, loss: 0.0697006806731224
epoch:  3, loss: 0.06147493049502373
epoch:  4, loss: 0.05360203608870506
epoch:  5, loss: 0.04699607565999031
epoch:  6, loss: 0.03137305751442909
epoch:  7, loss: 0.014018505811691284
epoch:  8, loss: 0.010648195631802082
epoch:  9, loss: 0.0071885003708302975
epoch:  10, loss: 0.006014049053192139
epoch:  11, loss: 0.005215431563556194
epoch:  12, loss: 0.004670539405196905
epoch:  13, loss: 0.004285433329641819
epoch:  14, loss: 0.003942360635846853
epoch:  15, loss: 0.0036864865105599165
epoch:  16, loss: 0.0034766795579344034
epoch:  17, loss: 0.0033113472163677216
epoch:  18, loss: 0.00317019852809608
epoch:  19, loss: 0.003039079485461116
epoch:  20, loss: 0.0029332186095416546
epoch:  21, loss: 0.0028336753603070974
epoch:  22, loss: 0.00274907355196774
epoch:  23, loss: 0.002718206262215972
epoch:  24, loss: 0.002629699418321252
epoch:  25, loss: 0.0025673522613942623

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9576641917228699
Test metrics:  R2 = 0.7119383811950684


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.37289878726005554
epoch:  1, loss: 0.3188565671443939
epoch:  2, loss: 0.31346458196640015
epoch:  3, loss: 0.3129311203956604
epoch:  4, loss: 0.31240129470825195
epoch:  5, loss: 0.3118748664855957
epoch:  6, loss: 0.2598155438899994
epoch:  7, loss: 0.2104269415140152
epoch:  8, loss: 0.1618097722530365
epoch:  9, loss: 0.13628682494163513
epoch:  10, loss: 0.11170010268688202
epoch:  11, loss: 0.09468402713537216
epoch:  12, loss: 0.08084043115377426
epoch:  13, loss: 0.0691608414053917
epoch:  14, loss: 0.05866102874279022
epoch:  15, loss: 0.04971017688512802
epoch:  16, loss: 0.04333033785223961
epoch:  17, loss: 0.04287106543779373
epoch:  18, loss: 0.042415015399456024
epoch:  19, loss: 0.04196454957127571
epoch:  20, loss: 0.04152562841773033
epoch:  21, loss: 0.03678080812096596
epoch:  22, loss: 0.03643007203936577
epoch:  23, loss: 0.03639550134539604
epoch:  24, loss: 0.036048613488674164
epoch:  25, loss: 0.03570877015590668
epoch:  26, loss: 0.0317938

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6281955242156982
Test metrics:  R2 = 0.4109612703323364


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="strong-wolfe"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5440237522125244
epoch:  1, loss: 0.5440237522125244
epoch:  2, loss: 0.5440237522125244
epoch:  3, loss: 0.5440237522125244
epoch:  4, loss: 0.5440237522125244
epoch:  5, loss: 0.5440237522125244
epoch:  6, loss: 0.5440237522125244
epoch:  7, loss: 0.5440237522125244
epoch:  8, loss: 0.5440237522125244
epoch:  9, loss: 0.5440237522125244
epoch:  10, loss: 0.5440237522125244
epoch:  11, loss: 0.5440237522125244
epoch:  12, loss: 0.5440237522125244
epoch:  13, loss: 0.5440237522125244
epoch:  14, loss: 0.5440237522125244
epoch:  15, loss: 0.5440237522125244
epoch:  16, loss: 0.5440237522125244
epoch:  17, loss: 0.5440237522125244
epoch:  18, loss: 0.5440237522125244
epoch:  19, loss: 0.5440237522125244
epoch:  20, loss: 0.5440237522125244
epoch:  21, loss: 0.5440237522125244
epoch:  22, loss: 0.5440237522125244
epoch:  23, loss: 0.5440237522125244
epoch:  24, loss: 0.5440237522125244
epoch:  25, loss: 0.5440237522125244
epoch:  26, loss: 0.5440237522125244
epoch:  27,

In [10]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -267536.1875
Test metrics:  R2 = -251193.96875


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    lr=1,
    c1=1e-4,
    c2=0.9,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="goldstein"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.4833826422691345
epoch:  1, loss: 0.4833826422691345
epoch:  2, loss: 0.4833826422691345
epoch:  3, loss: 0.4833826422691345
epoch:  4, loss: 0.4833826422691345
epoch:  5, loss: 0.4833826422691345
epoch:  6, loss: 0.4833826422691345
epoch:  7, loss: 0.4833826422691345
epoch:  8, loss: 0.4833826422691345
epoch:  9, loss: 0.4833826422691345
epoch:  10, loss: 0.4833826422691345
epoch:  11, loss: 0.4833826422691345
epoch:  12, loss: 0.4833826422691345
epoch:  13, loss: 0.4833826422691345
epoch:  14, loss: 0.4833826422691345
epoch:  15, loss: 0.4833826422691345
epoch:  16, loss: 0.4833826422691345
epoch:  17, loss: 0.4833826422691345
epoch:  18, loss: 0.4833826422691345
epoch:  19, loss: 0.4833826422691345
epoch:  20, loss: 0.4833826422691345
epoch:  21, loss: 0.4833826422691345
epoch:  22, loss: 0.4833826422691345
epoch:  23, loss: 0.4833826422691345
epoch:  24, loss: 0.4833826422691345
epoch:  25, loss: 0.4833826422691345
epoch:  26, loss: 0.4833826422691345
epoch:  27,

In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -117752.6484375
Test metrics:  R2 = -114217.6953125
